# Low-Rank Approximation (SVD factorization)

Replace a weight matrix `W` (m×n) with the product of two thin matrices `U` (m×r) and
`V` (r×n). Parameter count goes from `m·n` to `r·(m+n)`, and if `r` is small enough that
is a real saving in memory and in FLOPs.

This is the compression technique most often confused with something else, so it is worth
being blunt up front: **this is not LoRA**. [LoRA](lora-controlnet.ipynb) *adds* a
low-rank update to a frozen full-rank `W` (`W + BA`) to make fine-tuning cheap — the
deployed matrix is still full rank and the model is no smaller. Factorization *replaces*
`W` with a low-rank product, and the model really does shrink. Same linear algebra,
opposite purpose. Example 4 makes the distinction concrete.

Siblings: [Pruning](pruning.ipynb), [Sparsity induction](sparsity-induction.ipynb),
[Distillation](knowledge-distillation.ipynb), [Quantization](quantization-gptq-awq.ipynb).

## 1. What & Why

A trained weight matrix is usually not "full rank" in any useful sense. Its singular
values decay, often quickly, which means most of what the matrix does is captured by a
small number of directions. Truncating the rest is the most classical compression method
there is, and it comes with a guarantee no other technique in this library has:

> **Eckart–Young–Mirsky.** The rank-`r` truncated SVD is the *optimal* rank-`r`
> approximation to a matrix, in both the Frobenius and spectral norms. Not a heuristic —
> provably the best possible.

That guarantee is narrower than it sounds, and the gap is where most of the practical
difficulty lives: it is optimal at reconstructing **the weights**, and nobody cares about
the weights. What you care about is the layer's **output**, and the two objectives are
not the same (Example 3).

**Reach for it when:**

- A few large matrices dominate your parameter count — embedding tables, LM heads, the
  MLP up/down projections in a transformer.
- The spectrum actually decays. This is checkable in seconds and decides everything.
- You can afford a short fine-tune afterwards.

**Don't when:**

- The matrix is square-ish and you need a large rank to keep accuracy — you will *add*
  parameters (Example 2).
- The layer is memory-bandwidth-bound rather than compute-bound. Two small matmuls can
  be slower than one big one even with fewer FLOPs.

## 2. Mental Model

**The singular value spectrum is a bar chart of how much each "direction" of the
transformation matters**, sorted tallest first. A matrix whose bars fall off a cliff
after ten entries is really a rank-10 transformation wearing a large costume; keeping
all `n` of them is storing noise at full precision.

The analogue is lossy image compression. A photograph stored as raw pixels is `m×n`
numbers, but almost all of the perceptual content sits in a small number of frequency
components — so JPEG keeps those and throws the rest away. Truncated SVD is the same
move with the same guarantee, applied to a weight matrix instead of an image, and the
singular values are exactly the "how much does this component contribute" numbers.

The one place the analogy needs care: JPEG is tuned for the human eye, which is the
actual consumer of the output. Plain SVD is tuned for the Frobenius norm of the weights,
which is *not* the actual consumer. Making it activation-aware — weighting the
approximation by what the layer's inputs actually look like — is the fix, and it is the
single biggest practical improvement available here.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **SVD** | `W = U Σ Vᵀ`. `Σ` is diagonal, non-negative, sorted descending — the singular values. |
| **Truncated SVD** | Keep the top `r` singular values/vectors: `W_r = U_r Σ_r V_rᵀ`. |
| **Eckart–Young** | `W_r` is the provably optimal rank-`r` approximation in Frobenius/spectral norm. |
| **Break-even rank** | The rank below which factorization actually saves parameters: `r < mn/(m+n)`. For a square matrix, `n/2`. |
| **Energy / explained variance** | `Σᵢ₌₁ʳ σᵢ² / Σ σᵢ²` — the fraction of squared Frobenius norm retained at rank `r`. |
| **Effective rank** | The rank needed to retain some energy threshold (say 95%). The number that decides whether this technique applies at all. |
| **Spectral decay** | How fast `σ` falls. Fast decay → factorization works. Flat spectrum → it cannot. |
| **Activation-aware / data-aware SVD** | Factorize in a space weighted by the input covariance, minimising **output** error rather than weight error. (ASVD, FWSVD.) |
| **Per-layer rank allocation** | Distributing a global parameter budget across layers by their spectra rather than uniformly. |
| **Tensor decomposition** | The >2-D generalisation — Tucker and CP decompositions for convolution kernels. |
| **LoRA (contrast)** | `W + BA` with `W` frozen and full rank. A fine-tuning method, **not** a compression method. |

## 4. Setup

NumPy's `linalg.svd` is all that is needed — this is the one compression technique whose
core operation is a single library call.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
np.set_printoptions(precision=3, suppress=True)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — read the spectrum before you do anything else

Three matrices with identical shape and wildly different compressibility. The spectrum
tells you which is which in one line, and it is the only diagnostic that matters.

In [2]:
m = n = 256

# (a) genuinely low-rank plus a little noise
A = rng.standard_normal((m, 12)) @ rng.standard_normal((12, n)) + rng.standard_normal((m, n)) * 0.01
# (b) fast exponential decay -- the typical trained-layer spectrum
U0, _ = np.linalg.qr(rng.standard_normal((m, m)))
V0, _ = np.linalg.qr(rng.standard_normal((n, n)))
B = U0 @ np.diag(np.exp(-np.arange(n) / 20.0)) @ V0.T
# (c) a random dense matrix -- essentially incompressible
C = rng.standard_normal((m, n))

def energy_rank(W, keep=0.95):
    s = np.linalg.svd(W, compute_uv=False)
    cum = np.cumsum(s**2) / np.sum(s**2)
    return int(np.searchsorted(cum, keep) + 1)

print(f"{'matrix':28} {'r@90%':>7} {'r@95%':>7} {'r@99%':>7}  top singular values")
for name, W in [("(a) low-rank + noise", A), ("(b) exponential decay", B),
                ("(c) random dense", C)]:
    s = np.linalg.svd(W, compute_uv=False)
    print(f"{name:28} {energy_rank(W,0.90):7d} {energy_rank(W,0.95):7d} "
          f"{energy_rank(W,0.99):7d}  {np.round(s[:5], 2)}")

print(f"\nAll three are {m}x{n}. (a) and (b) compress; (c) needs "
      f"{energy_rank(C,0.95)} of {n} ranks to keep 95% -- there is nothing to remove.")
print("Running this check costs one SVD and saves you from factorizing a matrix that")
print("has no low-rank structure to exploit.")

matrix                         r@90%   r@95%   r@99%  top singular values
(a) low-rank + noise              11      11      12  [309.22 293.91 287.17 268.98 257.3 ]
(b) exponential decay             24      30      47  [1.   0.95 0.9  0.86 0.82]
(c) random dense                 131     157     198  [31.88 31.51 30.99 30.52 30.38]

All three are 256x256. (a) and (b) compress; (c) needs 157 of 256 ranks to keep 95% -- there is nothing to remove.
Running this check costs one SVD and saves you from factorizing a matrix that
has no low-rank structure to exploit.


### Example 2 — the break-even rank, and how easy it is to make the model bigger

Factorizing costs `r(m+n)` parameters against the original `mn`. That is only a saving
while `r < mn/(m+n)`. For a square matrix that is `n/2` — so a "compression" at rank
`0.75n` **adds 50% more parameters** while also losing accuracy.

In [3]:
def factorize(W, r):
    U, s, Vt = np.linalg.svd(W, full_matrices=False)
    return U[:, :r] * s[:r], Vt[:r]          # (m,r) and (r,n)

def report(W, r):
    Ur, Vr = factorize(W, r)
    orig, new = W.size, Ur.size + Vr.size
    err = np.linalg.norm(W - Ur @ Vr) / np.linalg.norm(W)
    return orig, new, new / orig, err

print(f"square {m}x{n}: break-even rank = {m*n//(m+n)}\n")
print(f"{'rank':>6} {'params':>10} {'vs orig':>9} {'rel err':>9}  verdict")
for r in (8, 16, 32, 64, 128, 160, 200):
    orig, new, ratio, err = report(B, r)
    verdict = "saves" if ratio < 1 else "COSTS MORE than the dense matrix"
    print(f"{r:6d} {new:10d} {ratio:8.2f}x {err:9.4f}  {verdict}")

print("\nNon-square matrices are far friendlier. A tall embedding table is the ideal case:")
# A realistic table: a low-rank core plus broadband noise, not an exactly-rank-8 matrix.
E = (rng.standard_normal((32000, 24)) @ rng.standard_normal((24, 512))
     + rng.standard_normal((32000, 512)) * 0.6)
print(f"  32000x512 embedding, break-even rank = {32000*512//(32000+512)}")
for r in (8, 32, 128, 256):
    orig, new, ratio, err = report(E, r)
    print(f"    rank {r:4d}: {ratio:.3f}x params, rel err {err:.4f}")
print("  Even at rank 256 -- half the width -- this table costs a quarter of the")
print("  original, because 32000 >> 512. Shape matters more than spectrum here.")

square 256x256: break-even rank = 128

  rank     params   vs orig   rel err  verdict
     8       4096     0.06x    0.6703  saves
    16       8192     0.12x    0.4493  saves
    32      16384     0.25x    0.2019  saves
    64      32768     0.50x    0.0408  saves
   128      65536     1.00x    0.0017  COSTS MORE than the dense matrix
   160      81920     1.25x    0.0003  COSTS MORE than the dense matrix
   200     102400     1.56x    0.0000  COSTS MORE than the dense matrix

Non-square matrices are far friendlier. A tall embedding table is the ideal case:
  32000x512 embedding, break-even rank = 503


    rank    8: 0.016x params, rel err 0.7700


    rank   32: 0.064x params, rel err 0.1167


    rank  128: 0.254x params, rel err 0.1021


    rank  256: 0.508x params, rel err 0.0810
  Even at rank 256 -- half the width -- this table costs a quarter of the
  original, because 32000 >> 512. Shape matters more than spectrum here.


### Example 3 — SVD minimises the wrong thing

Eckart–Young optimality is with respect to the **weights**. What a network cares about
is the **output**, `y = xW`. If some input directions carry far more energy than others —
and in real transformers they do, dramatically — then the best weight approximation is
not the best output approximation.

The fix is to factorize in a whitened space: weight the problem by the input covariance,
truncate there, and transform back.

In [4]:
d_in, d_out = 256, 256

# Inputs whose energy is concentrated in a few channels -- a stand-in for the
# well-documented outlier channels in transformer activations.
scales = np.logspace(1.5, -1.5, d_in)
X = rng.standard_normal((2048, d_in)) * scales
W = rng.standard_normal((d_in, d_out)) * 0.05

def svd_plain(W, r):
    U, s, Vt = np.linalg.svd(W, full_matrices=False)
    return (U[:, :r] * s[:r]) @ Vt[:r]

def svd_activation_aware(W, X, r, eps=1e-8):
    '''Factorize in a space whitened by the input second moment, so the truncation
    minimises output error rather than weight error.'''
    scale = np.sqrt((X ** 2).mean(axis=0)) + eps      # per-input-channel energy
    W_hat = W * scale[:, None]                        # move into the weighted space
    U, s, Vt = np.linalg.svd(W_hat, full_matrices=False)
    return ((U[:, :r] * s[:r]) @ Vt[:r]) / scale[:, None]     # ... and back

Y = X @ W
print(f"{'rank':>6} {'plain SVD':>22} {'activation-aware':>20}")
print(f"{'':6} {'weight err  output err':>22} {'output err':>20}")
for r in (8, 16, 32, 64, 128):
    Wp, Wa = svd_plain(W, r), svd_activation_aware(W, X, r)
    werr = np.linalg.norm(W - Wp) / np.linalg.norm(W)
    oerr_p = np.linalg.norm(Y - X @ Wp) / np.linalg.norm(Y)
    oerr_a = np.linalg.norm(Y - X @ Wa) / np.linalg.norm(Y)
    print(f"{r:6d} {werr:11.4f} {oerr_p:11.4f} {oerr_a:20.4f}")

print("\nPlain SVD wins on weight error by construction -- Eckart-Young guarantees it.")
print("It loses on output error, which is the only column that affects your model.")
print("Same rank, same parameter count, strictly better layer. This is the single")
print("highest-value change available in this notebook.")

  rank              plain SVD     activation-aware
       weight err  output err           output err
     8      0.9424      0.9376               0.7730
    16      0.8894      0.8793               0.6098
    32      0.7911      0.7810               0.3791
    64      0.6149      0.6127               0.1468
   128      0.3260      0.3257               0.0209

Plain SVD wins on weight error by construction -- Eckart-Young guarantees it.
It loses on output error, which is the only column that affects your model.
Same rank, same parameter count, strictly better layer. This is the single
highest-value change available in this notebook.


### Example 4 — factorization is not LoRA, and fewer FLOPs is not less time

Two closing points, both of which trip people up.

First the distinction from [LoRA](lora-controlnet.ipynb): the parameter counts move in
opposite directions. Second, the reason a FLOP reduction may not show up on a clock.

In [5]:
import time

W_full = rng.standard_normal((1024, 1024)).astype(np.float32)
r_lora = 16

print("--- what each technique leaves you holding at inference ---")
print(f"{'':34} {'params':>12} {'rank of the effective matrix':>30}")
print(f"{'original W':34} {W_full.size:12d} {'1024 (full)':>30}")
print(f"{'LoRA: W + BA (W frozen)':34} "
      f"{W_full.size + r_lora*2048:12d} {'1024 (still full)':>30}")
print(f"{'factorized: W -> U @ V, rank 16':34} {r_lora*2048:12d} {'16':>30}")
print("\nLoRA ADDS parameters and keeps the model the same size at deployment -- it")
print("buys cheap *training*. Factorization REPLACES W and the model gets smaller.")

print("\n--- FLOPs are not milliseconds ---")
X_b = rng.standard_normal((256, 1024)).astype(np.float32)
def bench(fn, reps=50):
    fn()
    t0 = time.perf_counter()
    for _ in range(reps): fn()
    return (time.perf_counter() - t0) / reps * 1e3

for r in (16, 64, 256, 400):
    U, V = factorize(W_full, r)
    U, V = U.astype(np.float32), V.astype(np.float32)
    flops_ratio = (r * (1024 + 1024)) / (1024 * 1024)
    t_dense = bench(lambda: X_b @ W_full)
    t_fact = bench(lambda: (X_b @ U) @ V)
    print(f"rank {r:4d}: {flops_ratio:5.2f}x FLOPs, {t_dense/t_fact:5.2f}x wall-clock "
          f"({t_dense:.2f}ms -> {t_fact:.2f}ms)")

print("\nThe speedup consistently lags the FLOP reduction: you now launch two kernels")
print("instead of one and materialise an intermediate (256 x r). At low rank the")
print("saving is large enough that it still wins handsomely; approaching break-even")
print("the theoretical 2x has vanished completely and you are marginally slower.")

--- what each technique leaves you holding at inference ---
                                         params   rank of the effective matrix
original W                              1048576                    1024 (full)
LoRA: W + BA (W frozen)                 1081344              1024 (still full)
factorized: W -> U @ V, rank 16           32768                             16

LoRA ADDS parameters and keeps the model the same size at deployment -- it
buys cheap *training*. Factorization REPLACES W and the model gets smaller.

--- FLOPs are not milliseconds ---


rank   16:  0.03x FLOPs,  4.51x wall-clock (0.21ms -> 0.05ms)


rank   64:  0.12x FLOPs,  2.75x wall-clock (0.21ms -> 0.07ms)


rank  256:  0.50x FLOPs,  1.01x wall-clock (0.21ms -> 0.20ms)


rank  400:  0.78x FLOPs,  1.01x wall-clock (0.21ms -> 0.20ms)

The speedup consistently lags the FLOP reduction: you now launch two kernels
instead of one and materialise an intermediate (256 x r). At low rank the
saving is large enough that it still wins handsomely; approaching break-even
the theoretical 2x has vanished completely and you are marginally slower.


## 6. Gotchas & Pitfalls

- **Factorizing past the break-even rank.** Example 2. At `r > mn/(m+n)` you have added
  parameters *and* lost accuracy. Compute the break-even rank before anything else; for
  square matrices it is `n/2` and good ranks are usually far below it.
- **Optimising weight error when you mean output error.** Example 3. Plain SVD is
  provably optimal for the wrong objective. Activation-aware factorization costs one
  calibration batch and is close to free.
- **Assuming fewer FLOPs means faster.** Two matmuls, an extra intermediate tensor, an
  extra kernel launch. Measure.
- **Not fine-tuning afterwards.** Truncation is a real perturbation. Even a short
  recovery pass on a small calibration set typically returns most of the loss — and
  unlike pruning, there is no mask to maintain, so it is an ordinary fine-tune.
- **Uniform rank across layers.** Spectra differ enormously between layers. Allocate a
  global budget by each layer's decay, not by a single global rank.
- **Factorizing attention projections naively.** Q/K/V often behave better factorized
  jointly (they share an input) than separately, and head structure means the matrix is
  not as unstructured as it looks.
- **Forgetting the bias.** `y = xW + b` factorizes as `y = (xU)V + b`. The bias is
  untouched — but it must still be there.
- **Confusing this with LoRA.** Example 4. If someone says "we used low-rank methods to
  compress the model" it is worth asking which of the two they mean, because one of them
  does not compress anything.

## 7. When to Use vs Alternatives

| Situation | Reach for |
|---|---|
| Big non-square matrices (embeddings, LM head, MLP projections) | **Low-rank factorization** |
| Square-ish matrices with a flat spectrum | Not this — try [quantization](quantization-gptq-awq.ipynb) |
| Cheapest overall win | [Quantization](quantization-gptq-awq.ipynb), every time |
| A genuinely smaller architecture | [Structured pruning](pruning.ipynb) or [distillation](knowledge-distillation.ipynb) |
| Cheap task adaptation, not compression | [LoRA / QLoRA](qlora.ipynb) — the opposite of this notebook |
| Convolutional kernels (4-D) | Tucker/CP tensor decomposition rather than plain matrix SVD |

**The honest position.** Low-rank factorization is the *least* used of the four
compression axes for language models, and the reason is Example 1: the weight matrices in
a well-trained transformer often have stubbornly flat spectra, so the effective rank is
high and there is little to truncate. Where it shines is the embedding table and output
head — huge, extremely non-square, and often genuinely low-rank — which on a small model
can be a third of all parameters.

It composes well: factorize, fine-tune briefly, then quantize the factors. And because it
produces a strictly smaller dense model with no sparsity pattern and no custom kernels,
whatever it does win is portable to any hardware, which is not true of pruning.

## 8. Resources

- [The approximation of one matrix by another of lower rank](https://link.springer.com/article/10.1007/BF02288367) — Eckart & Young, 1936. The optimality theorem.
- [numpy.linalg.svd](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html) — the API used throughout, including the `full_matrices` flag that matters for thin factors.
- [ASVD: Activation-aware Singular Value Decomposition for Compressing Large Language Models](https://arxiv.org/abs/2312.05821) — the Example 3 idea, developed properly.
- [Language model compression with weighted low-rank factorization](https://arxiv.org/abs/2207.00112) — FWSVD; Fisher-weighted importance instead of plain Frobenius.
- [Compressing Pre-trained Language Models by Matrix Decomposition](https://aclanthology.org/2020.aacl-main.88/) — a careful empirical study of where this does and does not work.
- [Tensor Decompositions and Applications](https://epubs.siam.org/doi/10.1137/07070111X) — Kolda & Bader; the standard reference for the >2-D case.
- [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685) — the technique this one is constantly confused with.

The Eckart-Young theorem guarantees the truncated SVD is the optimal rank-r approximation to a matrix. Why does that guarantee matter less in practice than it sounds?

In the JPEG analogy for truncated SVD, what plays the role of 'how much each component contributes', and what is the analogy's limitation?

Distinguish low-rank factorization from LoRA. Cover what each does to the weight matrix, what happens to the parameter count at deployment, and what each is actually for.

In [ ]:
def break_even_rank(m, n):
    ...


In [ ]:
assert break_even_rank(256, 256) == 127, break_even_rank(256, 256)
assert break_even_rank(1024, 1024) == 511
for m, n in [(256, 256), (1024, 1024), (32000, 512), (768, 3072)]:
    r = break_even_rank(m, n)
    assert r * (m + n) < m * n, (m, n, r, 'must save parameters')
    assert (r + 1) * (m + n) >= m * n, (m, n, r, 'must be the largest such r')
assert break_even_rank(32000, 512) == 503, break_even_rank(32000, 512)


You factorize a 1024×1024 layer at rank 256, cutting the FLOPs in half, and measure no wall-clock improvement at all. What is the most likely reason?

Which layers in a transformer are the best candidates for low-rank factorization and why? Explain the one-line diagnostic you should run first, and be honest about why this technique is the least used of the four compression axes for language models.